# Pipeline scratch/validation notebook

Loads a real GPS export (`data/20260901.geojson`) and walks the real `etl.py` / `praise_client.py` / `db.py` functions one stage at a time, printing the output of each stage before moving on. Mirrors exactly the sequence `main.py`'s `/ingest` endpoint runs internally, so the last cell's result is what `GET /exposure/hourly/{pid}` would return for the chart.

This notebook lives in `testing/`, one level below the project root where `etl.py`/`praise_client.py`/`db.py` actually live — the imports cell below adds the parent directory to `sys.path` to make that work.

`PRAISE_API_KEY`/`PRAISE_MYID` are read from the project-root `.env` automatically (via `python-dotenv` in `praise_client.py`) — fill in your real values there, nothing to edit in this notebook.

**Note:** Steps 3-8's cell outputs below are stale — captured before `etl.py` was changed to classify dwell runs before splitting by hour (fixing runs that straddle an HK-local clock hour being mislabelled). Re-run the whole notebook top to bottom to refresh them against the current pipeline.</cell_type>
</invoke>


In [1]:
import os

# PRAISE_API_KEY / PRAISE_MYID are loaded automatically from the project-root
# .env by praise_client.py (via python-dotenv) when it's imported below -
# nothing to hardcode here. Fill in your real values in ../.env.

# separate DB file so this notebook never touches the real exposure.db
os.environ["EXPOSURE_DB_PATH"] = "test_exposure.db"

In [2]:
import sys
from pathlib import Path
from pprint import pprint

# testing.ipynb lives in testing/, one level below the project root
# where etl.py / praise_client.py / db.py actually live.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import etl
import praise_client
import db

## Load real ping data

Reads the real GPS export at `data/20260901.geojson` (a GeoJSON `FeatureCollection`) — same shape `etl.parse_pings` expects (`properties.time`, `geometry.coordinates`), so it's used as-is.

In [3]:
import json

DATA_PATH = Path("data") / "20260901.geojson"  # relative to this notebook's own directory
PID = DATA_PATH.stem  # "20260901" - edit if you want a different pid

with open(DATA_PATH, encoding="utf-8") as f:
    geojson = json.load(f)

features = geojson["features"]
print(f"{len(features)} features loaded from {DATA_PATH}")
pprint(features[:3])

2253 features loaded from data\20260901.geojson
[{'geometry': {'coordinates': [114.2637567, 22.3377515], 'type': 'Point'},
  'properties': {'accuracy': 100.0,
                 'altitude': 137.10000610351562,
                 'provider': 'network',
                 'time': '2026-08-31T16:03:25.154Z',
                 'time_long': 1788192205154},
  'type': 'Feature'},
 {'geometry': {'coordinates': [114.2637567, 22.3377515], 'type': 'Point'},
  'properties': {'accuracy': 100.0,
                 'altitude': 137.10000610351562,
                 'provider': 'network',
                 'time': '2026-08-31T16:03:45.257Z',
                 'time_long': 1788192225257},
  'type': 'Feature'},
 {'geometry': {'coordinates': [114.2637617, 22.3377456], 'type': 'Point'},
  'properties': {'accuracy': 100.0,
                 'altitude': 137.10000610351562,
                 'provider': 'fused',
                 'time': '2026-08-31T16:07:30.262Z',
                 'time_long': 1788192450262},
  'type': 'Fe

## Step 1 — `etl.parse_pings`

GeoJSON → sorted `Ping` list.

In [4]:
pings = etl.parse_pings(features)

print(f"{len(pings)} pings parsed")
pprint(pings)

2253 pings parsed
[Ping(utc=datetime.datetime(2026, 8, 31, 16, 3, 25, 154000, tzinfo=datetime.timezone.utc),
      lng=114.2637567,
      lat=22.3377515),
 Ping(utc=datetime.datetime(2026, 8, 31, 16, 3, 45, 257000, tzinfo=datetime.timezone.utc),
      lng=114.2637567,
      lat=22.3377515),
 Ping(utc=datetime.datetime(2026, 8, 31, 16, 7, 30, 262000, tzinfo=datetime.timezone.utc),
      lng=114.2637617,
      lat=22.3377456),
 Ping(utc=datetime.datetime(2026, 8, 31, 16, 10, 31, 533000, tzinfo=datetime.timezone.utc),
      lng=114.2637537,
      lat=22.3377434),
 Ping(utc=datetime.datetime(2026, 8, 31, 16, 13, 32, 540000, tzinfo=datetime.timezone.utc),
      lng=114.2637608,
      lat=22.3377549),
 Ping(utc=datetime.datetime(2026, 8, 31, 16, 16, 33, 624000, tzinfo=datetime.timezone.utc),
      lng=114.26376,
      lat=22.3377518),
 Ping(utc=datetime.datetime(2026, 8, 31, 16, 19, 17, 774000, tzinfo=datetime.timezone.utc),
      lng=114.2637613,
      lat=22.3377517),
 Ping(utc=datetime.da

## Step 2 — `etl.remove_spike_outliers`

Deletes any point where the path sharply doubles back within a short time window. With real data the exact count removed isn't known ahead of time — inspect what got removed below.

In [5]:
cleaned = etl.remove_spike_outliers(pings)

print(f"before: {len(pings)} pings, after: {len(cleaned)} pings")

removed = [p for p in pings if p not in cleaned]
print(f"removed {len(removed)} spike(s):")
pprint(removed)

before: 2253 pings, after: 2176 pings
removed 77 spike(s):
[Ping(utc=datetime.datetime(2026, 8, 31, 18, 17, 27, 207000, tzinfo=datetime.timezone.utc),
      lng=114.26281971,
      lat=22.33968169),
 Ping(utc=datetime.datetime(2026, 8, 31, 18, 17, 28, 207000, tzinfo=datetime.timezone.utc),
      lng=114.26278538,
      lat=22.33960891),
 Ping(utc=datetime.datetime(2026, 8, 31, 18, 34, 55, 619000, tzinfo=datetime.timezone.utc),
      lng=114.2632989,
      lat=22.3399155),
 Ping(utc=datetime.datetime(2026, 8, 31, 18, 44, 59, 957000, tzinfo=datetime.timezone.utc),
      lng=114.2633021,
      lat=22.339894),
 Ping(utc=datetime.datetime(2026, 8, 31, 19, 11, 50, 229000, tzinfo=datetime.timezone.utc),
      lng=114.2632413,
      lat=22.3401437),
 Ping(utc=datetime.datetime(2026, 8, 31, 19, 44, 2, 619000, tzinfo=datetime.timezone.utc),
      lng=114.2632332,
      lat=22.3401468),
 Ping(utc=datetime.datetime(2026, 8, 31, 19, 53, 49, 400000, tzinfo=datetime.timezone.utc),
      lng=114.26322

## Step 3 — `etl.find_dwell_runs`

Classifies indoor/outdoor dwell runs over the full, continuous (unbucketed) ping stream, per the thresholds in `etl.py` (`DWELL_RADIUS_METERS`, `DWELL_INDOOR_MINUTES`). Done before any hour-splitting so a dwell run that straddles an HK-local clock hour is judged by its true total duration, not by whichever partial slice happens to land in one clock hour.

In [6]:
runs = etl.find_dwell_runs(cleaned)

print(f"{len(runs)} dwell run(s)")
for r in runs:
    print(f"{r.io:7s}  {r.start_utc} to {r.end_utc}  lng={r.lng:.5f} lat={r.lat:.5f}")

158 dwell run(s)
Indoor   2026-08-31 16:03:25.154000+00:00 to 2026-08-31 18:14:18.722000+00:00  lng=114.26375 lat=22.33775
Outdoor  2026-08-31 18:14:18.722000+00:00 to 2026-08-31 18:15:13.100000+00:00  lng=114.26332 lat=22.33767
Outdoor  2026-08-31 18:15:13.100000+00:00 to 2026-08-31 18:16:16.446000+00:00  lng=114.26306 lat=22.33790
Outdoor  2026-08-31 18:16:16.446000+00:00 to 2026-08-31 18:17:21.419000+00:00  lng=114.26281 lat=22.33870
Outdoor  2026-08-31 18:17:21.419000+00:00 to 2026-08-31 18:18:28.439000+00:00  lng=114.26299 lat=22.33978
Indoor   2026-08-31 18:18:28.439000+00:00 to 2026-09-01 00:14:33+00:00  lng=114.26324 lat=22.34007
Outdoor  2026-09-01 00:14:33+00:00 to 2026-09-01 00:15:36.008000+00:00  lng=114.26340 lat=22.33991
Indoor   2026-09-01 00:15:36.008000+00:00 to 2026-09-01 00:48:13+00:00  lng=114.26329 lat=22.33993
Indoor   2026-09-01 00:48:13+00:00 to 2026-09-01 01:29:42.777000+00:00  lng=114.26327 lat=22.33992
Outdoor  2026-09-01 01:29:42.777000+00:00 to 2026-09-01 0

## Step 4 — `etl.split_by_hour`

Slices each labelled run from Step 3 at HK-local clock-hour boundaries, carrying the run's `io` label into every hour it touches and prorating `delta_t_hours` per hour-slice. A run crossing N hours produces N same-labelled segments here.

In [ ]:
segments = [s for s in etl.split_by_hour(runs) if s.delta_t_hours > 0]

for s in segments:
    print(f"{s.io:7s}  hour={s.hour_start_hk}  start={s.start_time_hk}  delta_t_hours={s.delta_t_hours:.4f}  "
          f"lng={s.lng:.5f} lat={s.lat:.5f}")

## Step 5 — `etl.to_request_rows`

Builds the `[t, pid, lng, lat, x, delta_t]` rows `expo_calx` expects.

In [ ]:
rows = etl.to_request_rows(PID, segments)

pprint(rows)

## Step 6 — `praise_client.calculate_exposure`

The one real call to the third-party `expo_calx` API. Requires `PRAISE_API_KEY`/`PRAISE_MYID` set in the first cell.

In [ ]:
exposure_results = praise_client.calculate_exposure(rows)

pprint(exposure_results)

assert len(exposure_results) == len(rows), "expo_calx returned a different row count than requested"

## Step 7 — persist to SQLite

Same zip-and-build logic `main.py` uses (`-9` → `None` for a failed individual row).

In [ ]:
db.init_db()

db_rows = []
for seg, (_t, _pid, value) in zip(segments, exposure_results):
    db_rows.append({
        "pid": PID,
        "hour_start_hk": seg.hour_start_hk.strftime("%Y%m%d%H0000"),
        "io": seg.io,
        "delta_t_hours": seg.delta_t_hours,
        "exposure_value": None if value == -9 else value,
        "lng": seg.lng,
        "lat": seg.lat,
    })

pprint(db_rows)
db.upsert_hourly(db_rows)

## Step 8 — `db.get_hourly`

Read back what `GET /exposure/hourly/{pid}` would return for the chart.

In [ ]:
pprint(db.get_hourly(PID))

## Step 9 — verify gap-aware merge vs. the old count-based merge

Rebuilds `db_rows` with a dummy `exposure_value` (clustering never depends on
the scored value, only `io`/`lng`/`lat`/`start_time_hk`, so this avoids a real
`expo_calx` call on every sweep). Runs both the old fixed-100-chunk merge and
the new gap-aware merge (`main._merge_dense_rows`) on the same rows, prints
row counts, and confirms every consecutive same-`io` pair whose gap exceeds
`config.MAX_GAP_METERS`/`MAX_GAP_SECONDS` survives as two separate rows in the
new merge output — the direct proof that tunnel-like jumps are preserved
instead of disappearing into a merged chunk's middle.

Sweep `MAX_GAP_METERS`/`MAX_GAP_SECONDS`/`MAX_CLUSTER_SIZE` by editing
`config.py` (then re-running this cell) or by passing overrides straight into
`main._merge_dense_rows(db_rows, max_gap_meters=..., ...)` below. Re-run
against each file in `data/` by changing `DATA_PATH` in the Step 0 cell above
and re-running cells 4/6/8/10/12 before this one.

In [ ]:
import math
from datetime import datetime

import config  # same config.py main.py/etl.py import from

# main.py uses relative imports (`from . import config, db, etl, praise_client`),
# so it can't be bare-imported here the way etl/db/praise_client are - these
# mirror main.py's merge functions exactly (see main.py for the source of truth).

def _merge_chunk(chunk):
    last = chunk[-1]
    values = [r["exposure_value"] for r in chunk if r["exposure_value"] is not None]
    return {
        "pid": last["pid"], "hour_start_hk": last["hour_start_hk"],
        "start_time_hk": last["start_time_hk"], "io": last["io"],
        "delta_t_hours": sum(r["delta_t_hours"] for r in chunk),
        "exposure_value": sum(values) if values else None,
        "lng": last["lng"], "lat": last["lat"],
    }

def _old_merge_dense_rows(rows, chunk_size=100):
    merged, i, n = [], 0, len(rows)
    while i < n:
        io = rows[i]["io"]
        j = i
        while j < n and rows[j]["io"] == io:
            j += 1
        streak = rows[i:j]
        for k in range(0, len(streak), chunk_size):
            chunk = streak[k : k + chunk_size]
            merged.append(_merge_chunk(chunk)) if len(chunk) == chunk_size else merged.extend(chunk)
        i = j
    return merged

def _meters_between(lng1, lat1, lng2, lat2):
    x = math.radians(lng2 - lng1) * math.cos(math.radians(lat1)) * 6371000.0
    y = math.radians(lat2 - lat1) * 6371000.0
    return math.hypot(x, y)

def _parse_start_time_hk(s):
    return datetime.strptime(s, "%Y%m%d%H%M%S")

def _is_dense_pair(prev, curr, max_gap_meters, max_gap_seconds):
    if prev["io"] != curr["io"]:
        return False
    dt = abs((_parse_start_time_hk(curr["start_time_hk"]) - _parse_start_time_hk(prev["start_time_hk"])).total_seconds())
    if dt > max_gap_seconds:
        return False
    return _meters_between(prev["lng"], prev["lat"], curr["lng"], curr["lat"]) <= max_gap_meters

def _dense_clusters(rows, max_gap_meters, max_gap_seconds):
    clusters, i, n = [], 0, len(rows)
    while i < n:
        j = i + 1
        while j < n and _is_dense_pair(rows[j - 1], rows[j], max_gap_meters, max_gap_seconds):
            j += 1
        clusters.append(rows[i:j])
        i = j
    return clusters

def _new_merge_dense_rows(rows, max_gap_meters=config.MAX_GAP_METERS,
                           max_gap_seconds=config.MAX_GAP_SECONDS,
                           max_cluster_size=config.MAX_CLUSTER_SIZE):
    merged = []
    for cluster in _dense_clusters(rows, max_gap_meters, max_gap_seconds):
        for k in range(0, len(cluster), max_cluster_size):
            chunk = cluster[k : k + max_cluster_size]
            merged.append(_merge_chunk(chunk)) if len(chunk) == max_cluster_size else merged.extend(chunk)
    return merged


db_rows = [{
    "pid": PID,
    "hour_start_hk": seg.hour_start_hk.strftime("%Y%m%d%H0000"),
    "start_time_hk": seg.start_time_hk.strftime("%Y%m%d%H%M%S"),
    "io": seg.io,
    "delta_t_hours": seg.delta_t_hours,
    "exposure_value": 1.0,  # dummy - clustering doesn't depend on this
    "lng": seg.lng,
    "lat": seg.lat,
} for seg in segments]

old_merged = _old_merge_dense_rows(db_rows)
new_merged = _new_merge_dense_rows(db_rows)
print(f"{len(db_rows)} db_rows -> old merge: {len(old_merged)} rows, new merge: {len(new_merged)} rows")

new_merged_start_times = {r["start_time_hk"] for r in new_merged}
for prev, curr in zip(db_rows, db_rows[1:]):
    if prev["io"] != curr["io"]:
        continue
    dt = abs((_parse_start_time_hk(curr["start_time_hk"]) - _parse_start_time_hk(prev["start_time_hk"])).total_seconds())
    dist = _meters_between(prev["lng"], prev["lat"], curr["lng"], curr["lat"])
    if dt > config.MAX_GAP_SECONDS or dist > config.MAX_GAP_METERS:
        print(f"gap: {prev['start_time_hk']} -> {curr['start_time_hk']}  dt={dt:.0f}s dist={dist:.0f}m"
              f"  prev_survives={prev['start_time_hk'] in new_merged_start_times}"
              f"  curr_survives={curr['start_time_hk'] in new_merged_start_times}")